In [11]:
import os
import math
import time
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal

import imageio

from pettingzoo.sisl import waterworld_v4

In [ ]:
cfg = {
    # ---- ENV ----
    "n_pursuers": 8,   # nombre de pursuers
    "n_evaders": 5,    # nombre d'evaders
    "n_poisons": 10,   # nombre de poisons
    "n_coop": 2,       # nombre de pursuers cooperatifs pour pouvoir manger les evaders    
    "max_cycles": 500, # nombre de cycles par episode

    # ---- PPO ----
    "num_envs": 8,            # nombre d'environnements paralleles   
    "T": 512,                 # nombre de pas de temps par environnement avant mise a jour   
    "epochs": 4,              # nombre d'epochs par mise a jour  
    "minibatch_size": 512,    
    "gamma": 0.99,
    "gae_lambda": 0.95,
    "clip_ratio": 0.2,
    "target_kl": 0.015,

    # ---- Optim ----
    "actor_lr": 3e-4,        
    "critic_lr": 2e-5,       
    "max_grad_norm": 1.0,   

    # ---- Entropy schedule ----
    "ent_coef_start": 0.01,
    "ent_coef_end": 0.001,

    # ---- Training ----
    "num_updates": 500,
    "print_every": 20,
    "eval_every": 100,
    "eval_episodes": 5,

    # ---- Obs RMS ----
    "obs_rms_warmup_steps": 5000,

    # ---- Misc ----
    "seed": 0,
    "device": "cuda" if torch.cuda.is_available() else "cpu",
}

MODEL_PATH = "ppo_waterworld.pt"

In [13]:
def make_env(cfg, render_mode=None):
    env = waterworld_v4.parallel_env(
        n_pursuers=cfg["n_pursuers"],
        n_coop=cfg["n_coop"],
        max_cycles=cfg["max_cycles"],
        render_mode=render_mode,
    )
    return env

In [ ]:
def set_seeds(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def make_env(cfg, render_mode=None):
    return waterworld_v4.parallel_env(
        n_pursuers=cfg["n_pursuers"],
        n_evaders=cfg["n_evaders"],
        n_poisons=cfg["n_poisons"],
        n_coop=cfg["n_coop"],
        max_cycles=cfg["max_cycles"],
        render_mode=render_mode,
    )

def get_trainable_agents_from_possible(possible_agents: List[str]):
    return [a for a in possible_agents if "pursuer" in a]

def stack_obs(obs_dict: Dict[str, np.ndarray], agents: List[str], device: str):
    return torch.tensor(
        np.stack([obs_dict[a] for a in agents], axis=0),
        dtype=torch.float32,
        device=device

def unstack_actions(actions_tensor: torch.Tensor, agents: List[str]):
    # actions_tensor: [B, act_dim] -> dict agent->np


In [ ]:
class RunningMeanStd:
    def __init__(self, shape, eps=1e-4, device="cpu"):
        self.mean = torch.zeros(shape, dtype=torch.float32, device=device)
        self.var  = torch.ones(shape, dtype=torch.float32, device=device)
        self.count = torch.tensor(eps, dtype=torch.float32, device=device)

    @torch.no_grad()
    def update(self, x: torch.Tensor):
        if x.numel() == 0:
            return
        batch_mean = x.mean(dim=0)
        batch_var  = x.var(dim=0, unbiased=False)
        batch_count = torch.tensor(x.shape[0], dtype=torch.float32, device=x.device)

        delta = batch_mean - self.mean
        tot_count = self.count + batch_count

        new_mean = self.mean + delta * (batch_count / tot_count)
        m_a = self.var * self.count
        m_b = batch_var * batch_count
        M2 = m_a + m_b + (delta ** 2) * (self.count * batch_count / tot_count)
        new_var = M2 / tot_count

        self.mean = new_mean
        self.var = torch.clamp(new_var, min=1e-6)
        self.count = tot_count

def normalize_obs(x: torch.Tensor, rms: RunningMeanStd, clip=10.0):
    x = (x - rms.mean) / torch.sqrt(rms.var + 1e-8)
    return torch.clamp(x, -clip, clip)

In [16]:
class Actor(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
        )
        self.mean = nn.Linear(hidden, act_dim)
        self.logstd = nn.Parameter(torch.zeros(act_dim))

    def forward(self, obs):
        x = self.net(obs)
        mean = self.mean(x)
        std = torch.exp(self.logstd).unsqueeze(0).expand_as(mean)
        return mean, std

class Critic(nn.Module):
    def __init__(self, obs_dim, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, 1),
        )

    def forward(self, obs):
        return self.net(obs).squeeze(-1)

In [ ]:
def tanh_gaussian_sample(mean, std):
    dist = Normal(mean, std)
    u = dist.rsample()              
    a = torch.tanh(u)               # action in (-1,1)
    logp_u = dist.log_prob(u).sum(-1)
    corr = torch.log(1 - a.pow(2) + 1e-6).sum(-1)
    logp = logp_u - corr
    ent = dist.entropy().sum(-1)    
    return a, u, logp, ent

def tanh_gaussian_logp(mean, std, u, a):
    dist = Normal(mean, std)
    logp_u = dist.log_prob(u).sum(-1)
    corr = torch.log(1 - a.pow(2) + 1e-6).sum(-1)
    return logp_u - corr, dist.entropy().sum(-1)

def compute_gae(rew, done, v, v_last, gamma, lam):

    T, E, B = rew.shape
    adv = torch.zeros((T, E, B), device=rew.device)
    gae = torch.zeros((E, B), device=rew.device)
    v_next = v_last
    for t in reversed(range(T)):
        not_done = 1.0 - done[t]
        delta = rew[t] + gamma * not_done * v_next - v[t]
        gae = delta + gamma * lam * not_done * gae
        adv[t] = gae
        v_next = v[t].detach()
    ret = adv + v
    return ret, adv

@torch.no_grad()
def warmup_obs_rms(cfg, obs_rms: RunningMeanStd, device):
    print(f"Warmup obs RMS for {cfg['obs_rms_warmup_steps']} steps (random policy), then freeze.")
    envs = [make_env(cfg) for _ in range(cfg["num_envs"])]
    possible_agents = envs[0].possible_agents
    agents = get_trainable_agents_from_possible(possible_agents)

    obses = []
    steps = 0
    for e in range(cfg["num_envs"]):
        obs, _ = envs[e].reset(seed=cfg["seed"] + 1000 + e)
        obses.append(obs)

    while steps < cfg["obs_rms_warmup_steps"]:
        batch_obs = []
        for e in range(cfg["num_envs"]):
            env = envs[e]
            obs = obses[e]
            O = stack_obs(obs, agents, device)  
            batch_obs.append(O)
            # random actions in [-1,1]
            act_dim = env.action_space(agents[0]).shape[0]
            A = torch.rand((len(agents), act_dim), device=device) * 2 - 1
            actions = unstack_actions(A, agents)
            next_obs, rewards, term, trunc, _ = env.step(actions)

            all_done = True
            for a in agents:
                if not (term[a] or trunc[a]):
                    all_done = False
                    break
            if all_done:
                obs, _ = env.reset()
            else:
                obs = next_obs
            obses[e] = obs

        batch_obs = torch.cat(batch_obs, dim=0)  # [E*B, obs_dim]
        obs_rms.update(batch_obs)
        steps += 1

    for env in envs:
        env.close()

@torch.no_grad()
def rollout_multi_env(cfg, actor, critic, obs_rms, device):
    E = cfg["num_envs"]
    T = cfg["T"]

    envs = [make_env(cfg) for _ in range(E)]
    possible_agents = envs[0].possible_agents
    agents = get_trainable_agents_from_possible(possible_agents)
    B = len(agents)

    # reset
    obses = []
    for e in range(E):
        obs, _ = envs[e].reset(seed=cfg["seed"] + e)
        obses.append(obs)

    # episodic tracking
    ep_ret = [[0.0 for _ in range(B)] for _ in range(E)]
    finished_returns = []

    obs_dim = envs[0].observation_space(agents[0]).shape[0]
    act_dim = envs[0].action_space(agents[0]).shape[0]

    # buffers (all plain tensors, no graph)
    obs_buf  = torch.zeros((T, E, B, obs_dim), device=device)
    act_buf  = torch.zeros((T, E, B, act_dim), device=device)
    u_buf    = torch.zeros((T, E, B, act_dim), device=device)
    logp_buf = torch.zeros((T, E, B), device=device)
    ent_buf  = torch.zeros((T, E, B), device=device)
    v_buf    = torch.zeros((T, E, B), device=device)
    rew_buf  = torch.zeros((T, E, B), device=device)
    done_buf = torch.zeros((T, E, B), device=device)

    for t in range(T):
        for e in range(E):
            env = envs[e]
            obs = obses[e]

            O = stack_obs(obs, agents, device)       # [B, obs_dim]
            O = normalize_obs(O, obs_rms)

            mean, std = actor(O)
            A, U, logp, ent = tanh_gaussian_sample(mean, std)
            V = critic(O)

            actions = unstack_actions(A, agents)
            next_obs, rewards, term, trunc, _ = env.step(actions)

            r = torch.tensor([rewards[a] for a in agents], dtype=torch.float32, device=device)
            d = torch.tensor([float(term[a] or trunc[a]) for a in agents], dtype=torch.float32, device=device)

            # store (already no_grad, but we keep it explicit)
            obs_buf[t, e]  = O.detach()
            act_buf[t, e]  = A.detach()
            u_buf[t, e]    = U.detach()
            logp_buf[t, e] = logp.detach()
            ent_buf[t, e]  = ent.detach()
            v_buf[t, e]    = V.detach()
            rew_buf[t, e]  = r
            done_buf[t, e] = d

            for i in range(B):
                ep_ret[e][i] += float(r[i].item())

            all_done = all((term[a] or trunc[a]) for a in agents)
            if all_done:
                finished_returns.extend(ep_ret[e])
                ep_ret[e] = [0.0 for _ in range(B)]
                obs, _ = env.reset()
            else:
                obs = next_obs
            obses[e] = obs

    v_last = torch.zeros((E, B), device=device)
    for e in range(E):
        O_last = stack_obs(obses[e], agents, device)
        O_last = normalize_obs(O_last, obs_rms)
        v_last[e] = critic(O_last).detach()

    for env in envs:
        env.close()

    return (obs_buf, act_buf, u_buf, logp_buf, ent_buf, v_buf, rew_buf, done_buf, v_last,
            finished_returns, agents)

In [ ]:
def ppo_update(cfg, actor, critic, opt_actor, opt_critic,
               obs_buf, act_buf, u_buf, logp_old_buf, v_old_buf,
               ret_buf, adv_buf, device, ent_coef):

    T, E, B, obs_dim = obs_buf.shape
    act_dim = act_buf.shape[-1]
    N = T * E * B

    # flatten
    obs_flat = obs_buf.reshape(N, obs_dim)
    act_flat = act_buf.reshape(N, act_dim)
    u_flat   = u_buf.reshape(N, act_dim)
    logp_old = logp_old_buf.reshape(N).detach()
    v_old    = v_old_buf.reshape(N).detach()
    ret_flat = ret_buf.reshape(N).detach()
    adv_flat = adv_buf.reshape(N).detach()

    # normalize adv
    adv_flat = (adv_flat - adv_flat.mean()) / (adv_flat.std() + 1e-8)

    idx = torch.randperm(N, device=device)
    mb = cfg["minibatch_size"]
    clip_ratio = cfg["clip_ratio"]

    approx_kl = 0.0

    for epoch in range(cfg["epochs"]):
        for start in range(0, N, mb):
            mb_idx = idx[start:start+mb]

            O = obs_flat[mb_idx]
            A = act_flat[mb_idx]
            U = u_flat[mb_idx]

            mean, std = actor(O)
            logp_new, ent = tanh_gaussian_logp(mean, std, U, A)
            ratio = torch.exp(logp_new - logp_old[mb_idx])

            adv = adv_flat[mb_idx]
            surr1 = ratio * adv
            surr2 = torch.clamp(ratio, 1.0 - clip_ratio, 1.0 + clip_ratio) * adv
            actor_loss = -(torch.min(surr1, surr2)).mean()

            v_new = critic(O)
            critic_loss = 0.5 * ((v_new - ret_flat[mb_idx]) ** 2).mean()

            entropy_loss = -ent.mean()  
            loss_actor = actor_loss + ent_coef * entropy_loss

            # ---- backward actor ----
            opt_actor.zero_grad(set_to_none=True)
            loss_actor.backward()
            nn.utils.clip_grad_norm_(actor.parameters(), cfg["max_grad_norm"])
            opt_actor.step()

            # ---- backward critic ----
            opt_critic.zero_grad(set_to_none=True)
            critic_loss.backward()
            nn.utils.clip_grad_norm_(critic.parameters(), cfg["max_grad_norm"])
            opt_critic.step()

            with torch.no_grad():
                mb_kl = (logp_old[mb_idx] - logp_new).mean().item()
                approx_kl = mb_kl 
            if approx_kl > 1.5 * cfg["target_kl"]:
                return approx_kl, actor_loss.item(), critic_loss.item(), ent.mean().item(), True

    return approx_kl, actor_loss.item(), critic_loss.item(), ent.mean().item(), False

@torch.no_grad()
def evaluate(cfg, actor, critic, obs_rms, device, episodes=5):
    env = make_env(cfg)
    agents = get_trainable_agents_from_possible(env.possible_agents)

    returns = []
    for ep in range(episodes):
        obs, _ = env.reset(seed=cfg["seed"] + 999 + ep)
        ep_ret = {a: 0.0 for a in agents}

        while True:
            O = stack_obs(obs, agents, device)
            O = normalize_obs(O, obs_rms)
            mean, std = actor(O)
            A = torch.tanh(mean)  # deterministic
            actions = unstack_actions(A, agents)

            obs, rewards, term, trunc, _ = env.step(actions)
            for a in agents:
                ep_ret[a] += float(rewards[a])

            if all((term[a] or trunc[a]) for a in agents):
                break

        returns.extend([ep_ret[a] for a in agents])

    env.close()
    return float(np.mean(returns)) if len(returns) else float("nan")

def save_model(path, actor, critic, obs_rms):
    torch.save({
        "actor": actor.state_dict(),
        "critic": critic.state_dict(),
        "obs_rms_mean": obs_rms.mean.detach().cpu(),
        "obs_rms_var": obs_rms.var.detach().cpu(),
        "obs_rms_count": obs_rms.count.detach().cpu(),
    }, path)

def load_model(path, actor, critic, obs_rms, device):
    ckpt = torch.load(path, map_location=device, weights_only=False)
    actor.load_state_dict(ckpt["actor"])
    critic.load_state_dict(ckpt["critic"])
    obs_rms.mean = ckpt["obs_rms_mean"].to(device)
    obs_rms.var  = ckpt["obs_rms_var"].to(device)
    obs_rms.count = ckpt["obs_rms_count"].to(device)

def train(cfg, model_path=MODEL_PATH):
    device = cfg["device"]
    set_seeds(cfg["seed"])

    env0 = make_env(cfg)
    agents0 = get_trainable_agents_from_possible(env0.possible_agents)
    obs_dim = env0.observation_space(agents0[0]).shape[0]
    act_dim = env0.action_space(agents0[0]).shape[0]
    env0.close()

    print(f"obs_dim: {obs_dim} act_dim: {act_dim}")

    actor = Actor(obs_dim, act_dim).to(device)
    critic = Critic(obs_dim).to(device)

    opt_actor = optim.Adam(actor.parameters(), lr=cfg["actor_lr"])
    opt_critic = optim.Adam(critic.parameters(), lr=cfg["critic_lr"])

    obs_rms = RunningMeanStd(shape=(obs_dim,), device=device)

    warmup_obs_rms(cfg, obs_rms, device=device)

    for it in range(1, cfg["num_updates"] + 1):
        frac = it / cfg["num_updates"]
        ent_coef = cfg["ent_coef_start"] + frac * (cfg["ent_coef_end"] - cfg["ent_coef_start"])

        (obs_buf, act_buf, u_buf, logp_buf, ent_buf, v_buf, rew_buf, done_buf, v_last,
         finished_returns, agents) = rollout_multi_env(cfg, actor, critic, obs_rms, device)

        ret_buf, adv_buf = compute_gae(
            rew=rew_buf,
            done=done_buf,
            v=v_buf,
            v_last=v_last,
            gamma=cfg["gamma"],
            lam=cfg["gae_lambda"]
        )

        logp_old = logp_buf.detach()
        v_old = v_buf.detach()

        # update
        kl, aL, cL, ent_mean, early = ppo_update(
            cfg, actor, critic, opt_actor, opt_critic,
            obs_buf, act_buf, u_buf, logp_old, v_old,
            ret_buf, adv_buf,
            device=device, ent_coef=ent_coef
        )

        rollout_avg = rew_buf.sum(dim=0).mean().item() 
        train_ep_avg = float(np.mean(finished_returns)) if len(finished_returns) else float("nan")
        finished = len(finished_returns)

        if it % cfg["print_every"] == 0:
            msg = (f"it={it:4d}  train_ep_avg={train_ep_avg:8.3f}  rollout_avg={rollout_avg:8.3f}  "
                   f"ent={ent_mean:6.3f}  ent_coef={ent_coef:.4f}  kl={kl:.4f}  "
                   f"actorL={aL:6.3f}  criticL={cL:6.3f}  finished={finished}")
            if early:
                msg += "  (KL early stop)"
            print(msg)

        if it % cfg["eval_every"] == 0:
            eval_avg = evaluate(cfg, actor, critic, obs_rms, device, episodes=cfg["eval_episodes"])
            print(f"  | eval_avg={eval_avg:8.3f}")
            save_model(model_path, actor, critic, obs_rms)

    save_model(model_path, actor, critic, obs_rms)
    return actor, critic, obs_rms

In [ ]:
@torch.no_grad()
def play(cfg, model_path=MODEL_PATH, steps=2000):
    device = cfg["device"]

    env = make_env(cfg, render_mode="human")
    agents = get_trainable_agents_from_possible(env.possible_agents)

    obs_dim = env.observation_space(agents[0]).shape[0]
    act_dim = env.action_space(agents[0]).shape[0]

    actor = Actor(obs_dim, act_dim).to(device)
    critic = Critic(obs_dim).to(device)
    obs_rms = RunningMeanStd(shape=(obs_dim,), device=device)

    load_model(model_path, actor, critic, obs_rms, device=device)
    actor.eval(); critic.eval()

    obs, _ = env.reset(seed=cfg["seed"] + 4242)

    for t in range(steps):
        O = stack_obs(obs, agents, device)
        O = normalize_obs(O, obs_rms)

        mean, std = actor(O)
        A = torch.tanh(mean)  
        actions = unstack_actions(A, agents)

        obs, rewards, term, trunc, _ = env.step(actions)
        if all((term[a] or trunc[a]) for a in agents):
            obs, _ = env.reset()

    env.close()

In [20]:
@torch.no_grad()
def record_video(cfg, model_path=MODEL_PATH, video_path="waterworld.mp4", steps=3000, fps=30):
    device = cfg["device"]

    env = make_env(cfg, render_mode="rgb_array")
    agents = get_trainable_agents_from_possible(env.possible_agents)

    obs_dim = env.observation_space(agents[0]).shape[0]
    act_dim = env.action_space(agents[0]).shape[0]

    actor = Actor(obs_dim, act_dim).to(device)
    critic = Critic(obs_dim).to(device)
    obs_rms = RunningMeanStd(shape=(obs_dim,), device=device)

    load_model(model_path, actor, critic, obs_rms, device=device)
    actor.eval(); critic.eval()

    obs, _ = env.reset(seed=cfg["seed"] + 777)

    os.makedirs(os.path.dirname(video_path) or ".", exist_ok=True)
    writer = imageio.get_writer(video_path, fps=fps, macro_block_size=16)  # avoid codec issues

    for t in range(steps):
        frame = env.render()
        if frame is not None:
            writer.append_data(frame)

        O = stack_obs(obs, agents, device)
        O = normalize_obs(O, obs_rms)

        mean, std = actor(O)
        A = torch.tanh(mean)  # deterministic record (clean)
        actions = unstack_actions(A, agents)

        obs, rewards, term, trunc, _ = env.step(actions)
        if all((term[a] or trunc[a]) for a in agents):
            obs, _ = env.reset()

    writer.close()
    env.close()
    print(f"Saved video to: {video_path}")

In [ ]:
# 1. Train
actor, critic, obs_rms = train(cfg, model_path=MODEL_PATH)

# 2. Play
#play(cfg, model_path=MODEL_PATH, steps=3000)

# 3. Record
#record_video(cfg, model_path=MODEL_PATH, video_path="waterworld.mp4", steps=3000, fps=30)

obs_dim: 242 act_dim: 2
Warmup obs RMS for 5000 steps (random policy), then freeze.
it=  20  train_ep_avg= -95.273  rollout_avg= -97.837  ent= 2.464  ent_coef=0.0096  kl=0.0109  actorL=-0.045  criticL= 3.130  finished=64
it=  40  train_ep_avg= -86.199  rollout_avg= -88.599  ent= 2.111  ent_coef=0.0093  kl=0.0267  actorL=-0.050  criticL= 4.230  finished=64  (KL early stop)
it=  60  train_ep_avg= -72.391  rollout_avg= -74.678  ent= 1.719  ent_coef=0.0089  kl=0.0227  actorL=-0.076  criticL= 4.710  finished=64  (KL early stop)
it=  80  train_ep_avg= -63.493  rollout_avg= -64.944  ent= 1.339  ent_coef=0.0086  kl=0.0261  actorL=-0.047  criticL= 3.815  finished=64  (KL early stop)
it= 100  train_ep_avg= -58.748  rollout_avg= -60.205  ent= 0.978  ent_coef=0.0082  kl=0.0229  actorL=-0.002  criticL= 2.763  finished=64  (KL early stop)
  | eval_avg= -24.366
it= 120  train_ep_avg= -46.785  rollout_avg= -46.824  ent= 0.740  ent_coef=0.0078  kl=0.0257  actorL=-0.062  criticL= 2.883  finished=64  (KL

KeyboardInterrupt: 